# Nemotron-3-Nano SFT → GRPO ベースライン

## ワークフロー
1. **環境セットアップ**: Unsloth 4bit量子化でモデルロード
2. **データ準備**: CoTデータ読み込み + Train/Val層化分割
3. **SFT**: Chain-of-Thought データで教師ありファインチューニング
4. **GRPO**: 同一LoRAアダプタで強化学習による推論改善
5. **評価**: vLLM による推論 + 正答率計測
6. **提出**: submission.zip 作成

In [31]:
# ============================================================
# Config — 全ハイパーパラメータを一元管理
# ============================================================
import os


class Config:
    """全設定を一元管理するクラス。"""

    # --- 実行モード ---
    SEED = 42

    # --- モデル ---
    BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
    MODEL_PATH = None  # kagglehub でダウンロード後に設定

    # --- Unsloth 4bit 量子化 ---
    LOAD_IN_4BIT = True
    MAX_SEQ_LEN = 4096

    # --- LoRA ---
    LORA_RANK = 32
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "up_proj",
        "down_proj",
        "gate_proj",
        "in_proj",
        "out_proj",
    ]

    # --- データ ---
    # fold付き分割済みデータ（260328-nemotron-train-test-split で生成���
    TRAIN_FOLDS_CSV = "/kaggle/input/notebooks/yoshinarikawashima/260328-nemotron-train-test-split/train_with_folds.csv"
    # CoTデータの��スを指定
    COT_DATA_CSV = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv"
    # fold 0,1 = val (20%), fold 2-9 = train (80%)
    VAL_FOLDS = [0, 1]
    PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

    # --- SFT ---
    SFT_EPOCHS = 2
    SFT_BATCH_SIZE = 1
    SFT_GRAD_ACCUM = 8
    SFT_LR = 1e-4
    SFT_WARMUP_RATIO = 0.05
    SFT_LOGGING_STEPS = 10
    SFT_MAX_LENGTH = 4096
    SFT_PACKING = False

    # --- GRPO ---
    GRPO_EPOCHS = 1
    GRPO_BATCH_SIZE = 1
    GRPO_GRAD_ACCUM = 4
    GRPO_LR = 5e-6
    GRPO_NUM_GENERATIONS = 3
    GRPO_MAX_COMPLETION = 1024
    GRPO_TEMPERATURE = 0.7
    GRPO_BETA = 0.0  # 参照モデル不要
    GRPO_SUBSAMPLE_SIZE = 500
    GRPO_LOGGING_STEPS = 5

    # --- 評価 (vLLM) ---
    EVAL_MAX_TOKENS = 7680
    EVAL_TOP_P = 1.0
    EVAL_TEMPERATURE = 0.0
    EVAL_MAX_NUM_SEQS = 64
    EVAL_GPU_MEMORY_UTILIZATION = 0.85
    EVAL_MAX_MODEL_LEN = 8192

    # --- 出力 ---
    OUTPUT_DIR = "/kaggle/working/sft_grpo_output"
    SFT_ADAPTER_DIR = "/kaggle/working/sft_adapter"
    FINAL_ADAPTER_DIR = "/kaggle/working/final_adapter"

    # --- wandb (offline) ---
    WANDB_PROJECT = "nemotron-reasoning"
    WANDB_DIR = "/kaggle/working/wandb"
    WANDB_SFT_NAME = "sft"
    WANDB_GRPO_NAME = "grpo"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

print("Config defined.")


Config defined.


## 1. 環境セットアップ

In [34]:
# ============================================================
# 環境セットアップ — Triton/CUDA パッチ + 依存パッケージインストール
# ============================================================
import os, sys, subprocess, shutil, stat, glob, site, importlib.util

# --------------- ヘルパー ---------------


def _find_spec(name):
    """パッケージがインストール済みかチェックする。"""
    return importlib.util.find_spec(name) is not None


def _pick_best_wheel(wheels):
    """Python バージョンと PyTorch バージョンに合う wheel を選択する。"""
    import torch

    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    abi_tag = "cxx11abiTRUE" if torch.compiled_with_cxx11_abi() else "cxx11abiFALSE"
    exact = [w for w in wheels if py_tag in w and f"torch{torch_mm}" in w and abi_tag in w]
    if exact:
        return exact[-1]
    py_only = [w for w in wheels if py_tag in w]
    return py_only[-1] if py_only else None


def _pip_install(*args):
    """pip install をサブプロセスで実行する。失敗時は例外を送出する。"""
    subprocess.run([sys.executable, "-m", "pip", "install", *args], check=True)


# --------------- 1. Triton wheel インストール ---------------
candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if candidates:
    target = "/kaggle/working/pydeps"
    os.makedirs(target, exist_ok=True)
    _pip_install(
        "--no-deps",
        "--target",
        target,
        "--upgrade",
        "--ignore-installed",
        candidates[0],
    )
    if target not in sys.path:
        sys.path.insert(0, target)
    site.addsitedir(target)
    print(f"Triton installed from: {candidates[0]}")

# --------------- 2. ptxas バイナリの準備 ---------------
ptxas_src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"
ptxas_dst = "/tmp/ptxas-blackwell"

if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(
        ptxas_dst,
        os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH,
    )

    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(os.path.dirname(ptxas_src), dst_bin, dirs_exist_ok=True)

    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(
                fp,
                os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH,
            )

    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_dst
    os.environ["TRITON_PTXAS_PATH"] = ptxas_dst

    try:
        import triton.backends.nvidia as nv_backend

        nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")

        import triton.backends.nvidia.compiler as nv_compiler

        nv_compiler.get_ptxas_version = lambda arch: "12.0"
        print("Triton ptxas patches applied.")
    except ImportError:
        print("Triton not available yet, patches will be applied later.")

# --------------- 3. trl / datasets / mamba_ssm ---------------
OFFLINE_DIR = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
TRL_OFFLINE = "/kaggle/input/notebooks/johnnyhyland/offline-packages-trl/extra_package"

OFFLINE_SOURCES = {
    "datasets": OFFLINE_DIR,
    "trl": TRL_OFFLINE,
}

for pkg, find_links in OFFLINE_SOURCES.items():
    if not _find_spec(pkg):
        if not os.path.exists(find_links):
            raise FileNotFoundError(f"Offline package directory not found: {find_links}")
        _pip_install("-q", "--no-index", "--find-links", find_links, pkg)

if not _find_spec("mamba_ssm"):
    for pattern in [
        "/kaggle/input/**/*causal*conv1d*.whl",
        "/kaggle/input/**/*mamba_ssm*.whl",
    ]:
        wheels = sorted(glob.glob(pattern, recursive=True))
        best = _pick_best_wheel(wheels) if wheels else None
        if best:
            _pip_install("--no-index", "--no-deps", best)
        elif wheels:
            raise RuntimeError(f"No compatible wheel found for pattern: {pattern}")

import datasets, trl

print(f"datasets: {datasets.__version__}, trl: {trl.__version__}")

# --------------- 4. Unsloth ---------------
UNSLOTH_PACKAGES = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
if not os.path.exists(UNSLOTH_PACKAGES):
    raise FileNotFoundError(f"Unsloth package directory not found: {UNSLOTH_PACKAGES}")

_pip_install(
    "-q",
    "--no-index",
    "--find-links",
    UNSLOTH_PACKAGES,
    "--target",
    "/kaggle/working/packages",
    "--ignore-installed",
    "unsloth",
)

print("Environment setup done.")

Found Triton wheels: ['/kaggle/input/notebooks/johnnyhyland/offline-packages-trl/extra_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/notebooks/johnnyhyland/offline-packages-trl/extra_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Triton installed from: /kaggle/input/notebooks/johnnyhyland/offline-packages-trl/extra_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
datasets: 4.8.3, trl: 0.29.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you

Environment setup done.


## 2. モデルロード（Unsloth 4bit量子化）

In [14]:
# ============================================================
# RMSNorm パッチ + Unsloth でモデルロード
# ============================================================
import torch
import torch.nn.functional as F
import sys

sys.path.insert(0, '/kaggle/working/packages')

# --- Triton rmsnorm を純粋 PyTorch で差し替え ---
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    """RMSNorm の純粋 PyTorch 実装。Triton カーネルの代替。"""
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)


for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# --- Unsloth でモデルロード ---
from unsloth import FastLanguageModel
import kagglehub

Config.MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Model path: {Config.MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=Config.MODEL_PATH,
    max_seq_length=Config.MAX_SEQ_LEN,
    load_in_4bit=Config.LOAD_IN_4BIT,
    dtype=None,  # auto detect
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- CUDA カーネル問題回避 ---
for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

print(f"Model loaded in {'4bit' if Config.LOAD_IN_4BIT else 'bf16'} mode.")
print(f"Tokenizer vocab size: {len(tokenizer)}")

Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Nemotron_H does not support SDPA - switching to fast eager.


Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False
Model loaded in 4bit mode.
Tokenizer vocab size: 131072


In [15]:
# ============================================================
# LoRA アダプタの適用
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=Config.LORA_RANK,
    lora_alpha=Config.LORA_ALPHA,
    lora_dropout=Config.LORA_DROPOUT,
    target_modules=Config.LORA_TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=Config.SEED,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj', 'in_proj', 'out_proj']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


## 3. データ準備

In [35]:
# ============================================================
# データ読み込み・Train/Val 分割・CoT マージ
# ============================================================
import pandas as pd
import numpy as np
import re
import math

# --- fold付きデータ読み込み ---
train_full = pd.read_csv(Config.TRAIN_FOLDS_CSV)
print(f"Total samples: {len(train_full)}")
print(f"Columns: {list(train_full.columns)}")
print(f"\nTask type distribution:")
print(train_full['task_type'].value_counts())
print(f"\nFold distribution:")
print(train_full['fold'].value_counts().sort_index())

# --- Train / Val 分割（fold ベース、8:2） ---
val_df = train_full[train_full['fold'].isin(Config.VAL_FOLDS)].reset_index(drop=True)
train_df = train_full[~train_full['fold'].isin(Config.VAL_FOLDS)].reset_index(drop=True)

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}")

# --- CoT データのマージ（train のみ） ---
if os.path.exists(Config.COT_DATA_CSV):
    cot_df = pd.read_csv(Config.COT_DATA_CSV)
    print(f"\nCoT data loaded: {len(cot_df)} rows")
    if 'generated_cot' in cot_df.columns:
        train_df = train_df.merge(
            cot_df[['id', 'generated_cot']], on='id', how='left'
        )
        n_with_cot = train_df['generated_cot'].notna().sum()
        print(f"CoT merged: {n_with_cot} / {len(train_df)} train rows have CoT")
        # CoT が存在するもののみ SFT 学習データに使用
        sft_train_df = train_df[train_df['generated_cot'].notna()].reset_index(drop=True)
        print(f"SFT train data (CoT exists only): {len(sft_train_df)}")
    else:
        print("CoT data has no 'generated_cot' column.")
        sft_train_df = train_df.copy()
        sft_train_df['generated_cot'] = np.nan
else:
    print(f"\nCoT data not found at {Config.COT_DATA_CSV}")
    sft_train_df = train_df.copy()
    sft_train_df['generated_cot'] = np.nan

# --- 分割結果サマ�� ---
print(f"\n{'='*60}")
print(f"SFT Train: {len(sft_train_df)} (CoT available)")
print(f"GRPO Train: {len(train_df)} (full train split)")
print(f"Val: {len(val_df)} (fold {Config.VAL_FOLDS})")
print(f"{'='*60}")
print(f"\nSFT Train task distribution:")
print(sft_train_df['task_type'].value_counts())
print(f"\nVal task distribution:")
print(val_df['task_type'].value_counts())

Total samples: 9500
Columns: ['id', 'prompt', 'answer', 'task_type', 'is_correct', 'fold']

Task type distribution:
task_type
bit_manipulation        1602
gravitational           1597
unit_conversion         1594
encryption              1576
numeral_system          1576
transformation_rules    1555
Name: count, dtype: int64

Fold distribution:
fold
0    950
1    950
2    950
3    950
4    950
5    950
6    950
7    950
8    950
9    950
Name: count, dtype: int64

Train: 7600, Val: 1900

CoT data loaded: 6558 rows
CoT merged: 5245 / 7600 train rows have CoT
SFT train data (CoT exists only): 5245

SFT Train: 5245 (CoT available)
GRPO Train: 7600 (full train split)
Val: 1900 (fold [0, 1])

SFT Train task distribution:
task_type
gravitational           1216
numeral_system          1192
encryption              1127
unit_conversion         1074
bit_manipulation         481
transformation_rules     155
Name: count, dtype: int64

Val task distribution:
task_type
bit_manipulation        322
uni

In [36]:
# ============================================================
# ユーティリティ関数（正答判定 + 回答抽出）
# ============================================================
import re
import math


def verify(stored_answer, predicted):
    """正答判定: 数値は相対誤差1e-2以内、それ以外はcase-insensitive一致。"""
    stored_answer = str(stored_answer).strip()
    predicted = str(predicted).strip()
    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


def extract_final_answer(text):
    """モデル出力から最終回答を抽出する。"""
    if text is None:
        return 'NOT_FOUND'

    # \\boxed{} から抽出
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # フォールバック: "Final answer is: ..."
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    # 最後の数値
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]

    # 最後の非空行
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'


print("verify / extract_final_answer defined.")

verify / extract_final_answer defined.


In [39]:
# ============================================================
# SFT データセットの構築
# ============================================================
from datasets import Dataset as HFDataset
import re

BOXED_RE = re.compile(r'\\boxed\s*\{.*?\}', re.DOTALL)

def normalize_assistant_content(cot, answer):
    """
    generated_cot を安全に整形して、
    必ず
      <think> ... </think>
      \\boxed{answer}
    の形で返す。
    """
    answer = str(answer).strip()
    cot = "" if cot is None else str(cot).strip()

    # pandas由来の nan 対策
    if cot.lower() == "nan":
        cot = ""

    # 既存の boxed は消す（最後に answer を1回だけ付ける）
    cot = BOXED_RE.sub("", cot).strip()

    # 既存の <think>, </think> はいったん除去して中身だけ使う
    cot = re.sub(r"</?think>", "", cot).strip()

    # CoT が空ならデフォルト文
    if not cot:
        cot = "Let me analyze the pattern from the examples."

    assistant_content = f"<think>\n{cot}\n</think>\n\\boxed{{{answer}}}"
    return assistant_content


def build_sft_dataset(df, tokenizer):
    records = []

    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = row.get("generated_cot", "")

        user_content = prompt + Config.PROMPT_SUFFIX
        assistant_content = normalize_assistant_content(cot, answer)

        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        records.append({
            "text": text,
            "messages": messages,
        })

    ds = HFDataset.from_list(records)
    print(f"Built {len(ds)} SFT records")
    print(ds.column_names)
    return ds


# SFT 用データセット作成
sft_train_dataset = build_sft_dataset(sft_train_df, tokenizer)
sft_val_dataset = build_sft_dataset(val_df, tokenizer)

print(f"SFT Train dataset: {len(sft_train_dataset)}")
print(f"SFT Val dataset: {len(sft_val_dataset)}")

# --- 学習サンプル表示（SFT Dataset 構築後） ---
print(f"\n{'='*60}")
print("SFT 学習サンプル（各 task_type 1件ずつ）")
print(f"{'='*60}")

for task_type in sft_train_df["task_type"].unique():
    idx = sft_train_df[sft_train_df["task_type"] == task_type].index[0]
    sample = sft_train_dataset[idx]
    print(f"\n{'─'*60}")
    print(f"【task_type: {task_type}】")
    print(f"{'─'*60}")
    for msg in sample["messages"]:
        role = msg["role"].upper()
        content = msg["content"]
        if len(content) > 800:
            content = content[:800] + "\n... (truncated)"
        print(f"  [{role}]:\n{content}\n")


Built 5245 SFT records
['text', 'messages']
Built 1900 SFT records
['text', 'messages']
SFT Train dataset: 5245
SFT Val dataset: 1900

SFT 学習サンプル（各 task_type 1件ずつ）

────────────────────────────────────────────────────────────
【task_type: bit_manipulation】
────────────────────────────────────────────────────────────
  [USER]:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010001 -> 11011101
00001001 -> 01101101
00010101 -> 01010101
11111111 -> 10000001
10011101 -> 01000101
00111011 -> 00001001
10111101 -> 00000101
00100110 -> 10110011

Now, determine the output for: 00110100
Please put your final answer inside `\boxed{}`. For example: `\boxed{your answer}`

  [ASSISTANT]:
<think>
Using the per-bit analysis and the detected shift pattern, the most consistent output is:

- Bit 

## 4. SFT 学習

In [ ]:
# ============================================================
# wandb 初期化 (SFT)
# ============================================================
import os
import wandb

# --- W&B をローカル保存モードで設定する ---
os.environ["WANDB_MODE"] = "offline"
os.makedirs(Config.WANDB_DIR, exist_ok=True)

# Run を明示的に初期化する
wandb.init(
    project=Config.WANDB_PROJECT,
    name=Config.WANDB_SFT_NAME,
    dir=Config.WANDB_DIR,
    config={
        "phase": "SFT",
        "sft_epochs": Config.SFT_EPOCHS,
        "sft_batch_size": Config.SFT_BATCH_SIZE,
        "sft_grad_accum": Config.SFT_GRAD_ACCUM,
        "sft_lr": Config.SFT_LR,
        "sft_warmup_ratio": Config.SFT_WARMUP_RATIO,
        "sft_max_length": Config.SFT_MAX_LENGTH,
        "lora_rank": Config.LORA_RANK,
        "lora_alpha": Config.LORA_ALPHA,
        "lora_dropout": Config.LORA_DROPOUT,
    },
)

print(f"W&B のローカル保存先: {Config.WANDB_DIR}")
print(f"W&B Run 名: {Config.WANDB_SFT_NAME}")


In [42]:
# ============================================================
# SFT 学習の実行
# ============================================================
import gc
import time
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

# === Triton compiler fix ===
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"


class PrintLossCallback(TrainerCallback):
    """損失値を標準出力に表示するコールバック。"""

    def __init__(self, phase="SFT"):
        self.phase = phase
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"\n{'='*60}")
        print(f"[{self.phase}] Training started — {state.max_steps} steps")
        print(f"{'='*60}", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        elapsed = time.time() - self.start_time if self.start_time else 0
        loss = logs.get("loss", logs.get("train_loss", None))
        eval_loss = logs.get("eval_loss", None)
        lr = logs.get("learning_rate", None)
        parts = [f"[{self.phase}] step {state.global_step}/{state.max_steps}"]
        if loss is not None:
            parts.append(f"loss={loss:.4f}")
        if eval_loss is not None:
            parts.append(f"eval_loss={eval_loss:.4f}")
        if lr is not None:
            parts.append(f"lr={lr:.2e}")
        for key in ["reward", "reward_mean", "kl"]:
            if key in logs:
                parts.append(f"{key}={logs[key]:.4f}")
        parts.append(f"elapsed={elapsed/60:.1f}min")
        print(" | ".join(parts), flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start_time if self.start_time else 0
        print(f"\n[{self.phase}] Training complete — {state.global_step} steps in {elapsed/60:.1f}min", flush=True)


import wandb

# --- SFT Config ---
FastLanguageModel.for_training(model)

sft_config = SFTConfig(
    output_dir=Config.OUTPUT_DIR,
    num_train_epochs=Config.SFT_EPOCHS,
    per_device_train_batch_size=Config.SFT_BATCH_SIZE,
    per_device_eval_batch_size=Config.SFT_BATCH_SIZE,
    gradient_accumulation_steps=Config.SFT_GRAD_ACCUM,
    learning_rate=Config.SFT_LR,
    lr_scheduler_type="cosine",
    warmup_ratio=Config.SFT_WARMUP_RATIO,
    max_length=Config.SFT_MAX_LENGTH,
    logging_steps=Config.SFT_LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=Config.SEED,
    report_to="wandb",
    run_name=Config.WANDB_SFT_NAME,
    packing=Config.SFT_PACKING,
    dataset_text_field="text", 
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_train_dataset,
    eval_dataset=sft_val_dataset,
    processing_class=tokenizer,
    callbacks=[PrintLossCallback("SFT")],
)

print(
    f"SFT config: epochs={Config.SFT_EPOCHS}, bs={Config.SFT_BATCH_SIZE}, grad_accum={Config.SFT_GRAD_ACCUM}, lr={Config.SFT_LR}"
)
print("Starting SFT training...")
sft_trainer.train()

# --- SFT アダプタ保存 ---
os.makedirs(Config.SFT_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(Config.SFT_ADAPTER_DIR)
tokenizer.save_pretrained(Config.SFT_ADAPTER_DIR)
print(f"SFT adapter saved to {Config.SFT_ADAPTER_DIR}")

wandb.finish()
print("wandb SFT run finished.")

del sft_trainer
gc.collect()
torch.cuda.empty_cache()

Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/5245 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/1900 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFT config: epochs=2, bs=1, grad_accum=8, lr=0.0001
Starting SFT training...

[SFT] Training started — 1312 steps


RuntimeError: index_add_(): self (BFloat16) and source (Float) must have the same scalar type

## 5. GRPO 学習

In [ ]:
# ============================================================
# wandb 初期化 (GRPO)
# ============================================================
import wandb

# Run を明示的に初期化する
wandb.init(
    project=Config.WANDB_PROJECT,
    name=Config.WANDB_GRPO_NAME,
    dir=Config.WANDB_DIR,
    config={
        "phase": "GRPO",
        "grpo_epochs": Config.GRPO_EPOCHS,
        "grpo_batch_size": Config.GRPO_BATCH_SIZE,
        "grpo_grad_accum": Config.GRPO_GRAD_ACCUM,
        "grpo_lr": Config.GRPO_LR,
        "grpo_num_generations": Config.GRPO_NUM_GENERATIONS,
        "grpo_max_completion": Config.GRPO_MAX_COMPLETION,
        "grpo_temperature": Config.GRPO_TEMPERATURE,
        "grpo_beta": Config.GRPO_BETA,
        "lora_rank": Config.LORA_RANK,
        "lora_alpha": Config.LORA_ALPHA,
    },
)

print(f"W&B のローカル保存先: {Config.WANDB_DIR}")
print(f"W&B Run 名: {Config.WANDB_GRPO_NAME}")


In [ ]:
# ============================================================
# GRPO 報酬関数の定義
# ============================================================


def reward_fn(completions, answer, **kwargs):
    """GRPO 用報酬関数: 正解+1.0、不正解-0.5、NOT_FOUND -1.0。"""
    rewards = []
    for completion, ans in zip(completions, answer):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        extracted = extract_final_answer(text)

        if extracted == 'NOT_FOUND':
            rewards.append(-1.0)
        elif verify(str(ans), extracted):
            rewards.append(1.0)
        else:
            rewards.append(-0.5)

    return rewards


def format_reward_fn(completions, **kwargs):
    """\\boxed{} フォーマットの有無で報酬を与える。"""
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        if re.search(r'\\boxed\{[^}]+\}', text):
            rewards.append(0.5)
        else:
            rewards.append(-0.5)
    return rewards


print("Reward functions defined.")

In [ ]:
# ============================================================
# GRPO 学習の実行
# ============================================================
from trl import GRPOTrainer, GRPOConfig


FastLanguageModel.for_training(model)  # 学習モードに再切替

grpo_config = GRPOConfig(
    output_dir=Config.OUTPUT_DIR + "/grpo",
    num_train_epochs=Config.GRPO_EPOCHS,
    per_device_train_batch_size=Config.GRPO_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRPO_GRAD_ACCUM,
    learning_rate=Config.GRPO_LR,
    lr_scheduler_type="cosine",
    logging_steps=Config.GRPO_LOGGING_STEPS,
    save_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=Config.SEED,
    report_to="wandb",
    run_name=Config.WANDB_GRPO_NAME,
    # GRPO specific
    num_generations=Config.GRPO_NUM_GENERATIONS,
    max_completion_length=Config.GRPO_MAX_COMPLETION,
    temperature=Config.GRPO_TEMPERATURE,
    beta=Config.GRPO_BETA,
    max_prompt_length=Config.SFT_MAX_LENGTH,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    reward_funcs=[reward_fn, format_reward_fn],
    callbacks=[PrintLossCallback("GRPO")],
)

print(f"GRPO config: {Config.GRPO_NUM_GENERATIONS} generations/prompt, temp={Config.GRPO_TEMPERATURE}, lr={Config.GRPO_LR}")
print("Starting GRPO training (warm-started from SFT)...")
grpo_trainer.train()

# --- 最終アダプタを保存 ---
os.makedirs(Config.FINAL_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(Config.FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(Config.FINAL_ADAPTER_DIR)
print(f"Final adapter (SFT+GRPO) saved to {Config.FINAL_ADAPTER_DIR}")

wandb.finish()
print("wandb GRPO run finished.")

del grpo_trainer
gc.collect()
torch.cuda.empty_cache()

## 6. vLLM による評価

In [ ]:
# ============================================================
# メモリ解放 + モデルキャッシュ
# ============================================================
import gc
import time
import multiprocessing
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Unsloth モデルを解放して vLLM 用に GPU を空ける ---
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared for vLLM.")


# --- モデルキャッシュ（ページキャッシュにプリロード） ---
def cache_model(path, exts=('.bin', '.pt', '.safetensors'), num_workers=None, chunk_mb=256):
    """モデルファイルを OS ページキャッシュに読み込んで後続のロードを高速化する。"""
    def warmup_file(fpath):
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, 'rb') as f:
                while True:
                    data = f.read(chunk_size)
                    if not data:
                        break
                    total += len(data)
        except Exception as e:
            print(f'Error reading {fpath}: {e}')
        return fpath, total

    path = Path(path)
    files = sorted(p for p in path.rglob('*') if p.is_file() and str(p).endswith(exts)) if path.is_dir() else []
    if not files:
        return 0

    if num_workers is None:
        num_workers = min(multiprocessing.cpu_count(), 8)

    t0 = time.time()
    total_bytes = 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n

    elapsed = time.time() - t0
    print(f'[cache_model] {len(files)} files, {total_bytes/1024**3:.2f} GB in {elapsed:.1f}s')
    return total_bytes


cache_model(Config.MODEL_PATH, num_workers=16, chunk_mb=1024)

In [ ]:
# ============================================================
# vLLM 推論 + 正答率計測
# ============================================================
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# --- vLLM エンジン初期化 ---
llm = LLM(
    model=str(Config.MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=Config.EVAL_MAX_NUM_SEQS,
    gpu_memory_utilization=Config.EVAL_GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=Config.EVAL_MAX_MODEL_LEN,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=Config.LORA_RANK,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)

sampling_params = SamplingParams(
    temperature=Config.EVAL_TEMPERATURE,
    top_p=Config.EVAL_TOP_P,
    max_tokens=Config.EVAL_MAX_TOKENS,
)

vllm_tokenizer = llm.get_tokenizer()

# --- 評価対象: val_df ---
print(f"Evaluating on {len(val_df)} validation samples...")

prompts = []
for item in val_df.itertuples(index=False):
    user_content = item.prompt + Config.PROMPT_SUFFIX
    try:
        prompt = vllm_tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except Exception:
        prompt = user_content
    prompts.append(prompt)

# --- 推論実行 ---
LORA_PATH = Config.FINAL_ADAPTER_DIR

outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest("adapter", 1, LORA_PATH),
)

# --- 結果集計 ---
results = []
num_correct = 0
for item, output in zip(val_df.itertuples(index=False), outputs):
    raw_text = output.outputs[0].text
    extracted = extract_final_answer(raw_text)
    is_correct = verify(str(item.answer), extracted)
    if is_correct:
        num_correct += 1
    results.append({
        'id': item.id,
        'task_type': item.task_type,
        'answer': item.answer,
        'prediction': extracted,
        'is_correct': is_correct,
        'raw_output': raw_text,
    })

results_df = pd.DataFrame(results)
results_df.to_csv('/kaggle/working/val_results.csv', index=False)

# --- 全体正答率 ---
accuracy = num_correct / len(val_df)
print(f"\n{'='*60}")
print(f"Overall Accuracy: {accuracy:.4f}  ({num_correct}/{len(val_df)})")
print(f"{'='*60}")

# --- タスクタイプ別正答率 ---
print("\nAccuracy by task type:")
type_acc = results_df.groupby('task_type')['is_correct'].agg(['mean', 'sum', 'count'])
type_acc.columns = ['accuracy', 'correct', 'total']
print(type_acc.to_string())

# --- vLLM 解放 ---
del llm
gc.collect()
torch.cuda.empty_cache()

## 7. 提出ファイル作成

In [ ]:
# ============================================================
# submission.zip の作成
# ============================================================
import json
import shutil
import zipfile

SUBMISSION_DIR = "/kaggle/working/submission_adapter"
os.makedirs(SUBMISSION_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

for fname in required_files:
    src = os.path.join(Config.FINAL_ADAPTER_DIR, fname)
    dst = os.path.join(SUBMISSION_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

# --- adapter_config.json の修正（提出用） ---
config_path = os.path.join(SUBMISSION_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = Config.BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

# --- ZIP 作成 ---
zip_path = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")

# --- 整合性チェック ---
with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("submission.zip ready to submit!")